# Analyse du Data Drift — Projet P08

Analyse de la dérive des données de production de l'API de scoring crédit de l'entreprise « Prêt à Dépenser », réalisée avec **Evidently**.

- **Référence** : échantillon stratifié de 10 000 lignes des données d'entraînement (`data/reference_sample.parquet`).
- **Production** : champ `inputs` extrait du journal des prédictions (`logs/predictions.jsonl`).

Deux comparaisons sont produites pour montrer qu'Evidently détecte bien la dérive lorsqu'elle est présente :
1. référence vs **toute la production** (1 000 requêtes) ;
2. référence vs la **vague dérivée** seule (les 500 dernières, scénario « récession »).

## 1. Chargement des données

In [ ]:
from pathlib import Path
import pandas as pd
from evidently import Report
from evidently.presets import DataDriftPreset

# Racine du projet : on remonte jusqu'au dossier contenant data/
ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

REF_FILE = ROOT / 'data' / 'reference_sample.parquet'
LOG_FILE = ROOT / 'logs' / 'predictions.jsonl'
print('Racine projet :', ROOT)

In [ ]:
# Référence : on retire TARGET (le drift de DONNÉES porte sur les features)
reference = pd.read_parquet(REF_FILE).drop(columns=['TARGET'], errors='ignore')
reference.shape

In [ ]:
# Production : le champ `inputs` (un dict par ligne) -> DataFrame de features
log = pd.read_json(LOG_FILE, lines=True)
log['timestamp'] = pd.to_datetime(log['timestamp'])
log = log.sort_values('timestamp').reset_index(drop=True)

prod_all = pd.json_normalize(log['inputs'].tolist())
prod_drift = prod_all.tail(500)
prod_all.shape, prod_drift.shape

### Contrôle du découpage des vagues

Le journal contient deux vagues : 500 requêtes normales, puis 500 « dérivées ». On vérifie l'écart de probabilité moyenne, qui doit être net.

In [ ]:
print('proba moyenne 500 premières (normale) :', round(log['proba_defaut'].iloc[:500].mean(), 3))
print('proba moyenne 500 dernières (dérivée) :', round(log['proba_defaut'].iloc[500:].mean(), 3))

## 2. Fonction de rapport de drift

On compare uniquement les colonnes communes (les 21 features du noyau journalisées), en écartant celles entièrement vides d'un côté.

> **Note API Evidently 0.7.x** : `report.run(current, reference)` prend la **production en premier**, la référence ensuite.

In [ ]:
def rapport_drift(reference: pd.DataFrame, current: pd.DataFrame):
    common = [c for c in reference.columns if c in current.columns]
    valides = [c for c in common
               if reference[c].notna().any() and current[c].notna().any()]
    report = Report([DataDriftPreset()])
    # ordre 0.7.x : (current, reference)
    return report.run(current[valides], reference[valides])

## 3. Comparaison 1 — référence vs toute la production

Mélange des deux vagues : le drift y est *moyenné*, donc atténué.

In [ ]:
res_all = rapport_drift(reference, prod_all)
res_all  # affichage interactif du rapport dans le notebook
# Pour exporter en HTML : res_all.save_html('rapport_drift_all.html')

## 4. Comparaison 2 — référence vs vague dérivée

La vague la plus déviante (scénario récession) : le drift y est fort et concentré sur les features perturbées.

In [ ]:
res_drift = rapport_drift(reference, prod_drift)
res_drift
# Pour exporter en HTML : res_drift.save_html('rapport_drift_vague_derivee.html')

## 5. Interprétation

### Drift par feature (vague dérivée)

Le drift se concentre, comme attendu, sur les features perturbées par le scénario (revenus, crédits, annuités, scores externes). Scores de drift les plus élevés (distance de Wasserstein normée) : `EXT_SOURCE_3` ≈ 1,37 · `EXT_SOURCE_2` ≈ 1,30 · `EXT_SOURCE_1` ≈ 1,23 · `AMT_INCOME_TOTAL` ≈ 0,65 · `AMT_CREDIT` ≈ 0,61 · `AMT_ANNUITY` ≈ 0,38. Les features non perturbées (`CODE_GENDER`, `FLAG_OWN_CAR`, `DAYS_EMPLOYED`…) restent non driftées : Evidently isole correctement le signal.

### Le paradoxe du verdict global

| Comparaison | Colonnes driftées | Part | Verdict « Dataset Drift » |
|---|---|---|---|
| Vague dérivée | 10 / 21 | 0,476 | **NON détecté** |
| Toute la production | 11 / 21 | 0,524 | **détecté** |

Point central : la vague **la plus déviante** (scores jusqu'à 1,37) n'est **pas** flaggée au niveau dataset, alors que le mélange, moins intense, l'est. La raison : le verdict global d'Evidently repose sur la **proportion de colonnes** driftées (seuil 0,5), et non sur l'**intensité** du drift. Sur la vague dérivée, le drift est très fort mais concentré sur 10 features (0,476, juste sous la barre des 50 %).

### Conséquences opérationnelles

1. **Ne jamais se fier au seul verdict binaire** « Dataset Drift detected » : un drift à fort impact peut passer sous le seuil de proportion.
2. **Analyser le drift par feature** et le **pondérer par l'importance** des variables : ici, les 3 features les plus driftées (`EXT_SOURCE_*`) sont aussi les 3 plus importantes du modèle — le drift le plus dangereux est précisément celui que le verdict global masque.
3. **Corréler avec la sortie du modèle** : la proba de défaut moyenne double (0,358 → 0,697) sur la vague dérivée — le drift d'entrée se propage en drift de prédiction.

> Donnée Home Credit fictive et publique. En production réelle, la journalisation des inputs imposerait pseudonymisation, minimisation et durée de conservation limitée (RGPD).